![](module-1/img/KurrawongAI_350.png)

# Address Model Training - Module 3, Use & Maintenance

## Outline

## 1. Use

### 1.1 Joining to other data

#### 1.1.1 Cadastral data

Joining to Cadastral data is catered for in one way by the model only:

```mermaid
flowchart LR
    A[Address] -->|is name for| B[Addressable Object]
```

An `Address` is an instance of the class `https://linked.data.gov.au/def/addr/Address`, e.g.:

```turtle
PREFIX addr: <https://linked.data.gov.au/def/addr/>

<http://example.com/123-456-789> a addr:Address .
```

An `Addressable Object` is anything that an `Address` can be assigned to, e.g. a `Parcel` or a `Property` from the [Cadastral Model](https://linked.data.gov.au/def/cad).

The predicate used to link to a cadastral object is known: `cn:isNameFor`, so:

```turtle
PREFIX cn: <https://linked.data.gov.au/def/cn/>

<http://example.com/123-456-789> cn:isNameFor <SOME-ADDRESSABLE-OBJECT-IRI> .
```

##### Universal identifiers

The critical thing here is to use a form of identifier for the `Addressable Object` that the model (or validators for it) understand. This means using an [IRI - Internationalised Resource Identifier](https://en.wikipedia.org/wiki/Internationalized_Resource_Identifier): a [URI](https://en.wikipedia.org/wiki/Uniform_Resource_Identifier) that allows non-ASCII characters.

This enables no ambiguity of object identity across multiple datasets and even jurisdictions.

If we have a local ID for an Addressable Object such as a Parcel like `P987654321`, we need to "universalise" it by joining this local ID to a globally-uniq namespace.

Qld uses `https://linked.data.gov.au/dataset/qld-addr/` + type name, e.g. `parcel`, giving something like `https://linked.data.gov.au/dataset/qld-addr/parcel/33RP171651`.

Imagine NSW uses `https://linked.data.gov.au/dataset/nsw-cad/` you may have:

* `https://linked.data.gov.au/dataset/nsw-cad/P987654321`, or
* `https://linked.data.gov.au/dataset/nsw-cad/parcel/987654321`

##### Identifier system

The `linked.data.gov.au` [Australian Government Linked Data Working Group](https://www.linked.data.gov.au)'s Linked Data identifier system was set up to provide identifiers for Linked Data objects to all Australian government agencies (and friends), for free. It is used for the Address Model - <https://linked.data.gov.au/def/addr> and for many datasets used with the model, such as the Qld one mentioned above.

##### Identifier registration

Identifiers to be used in production should be registered within the [AGLDWG's Catalogue of PIDs](https://catalogue.linked.data.gov.au).

#### 1.1.2 Place Names

Address data may link to Geographical (Place) Names is two ways in the model:

* **direct**: as objects referenced in address parts within an Address
* **indirect**: via `Address --isNameFor--> AddressableObject` where also `GeographicalName --isNameFor--> AddressableObject`
    * e.g. "Parliament Drive, Canberra ACT 2600, Australia": "Parliament House"

We may only declare **direct** links in address data but **indirect** links will occur when (if?) consistent Addressable Object / Geographical Object identifiers are used.

Imagine a house with a registered place name of "Haunted House" and an address of "72 Yundah St, Shorncliffe...":

```turtle
<http://example.com/address/hh>
    a addr:Address ;
    schema:hasPart [
        schema:additionalType apt:propertyName ;
        schema:value <https://linked.data.gov.au/dataset/qldgn/QLD12345> ; # "Yundah St"
    ] ,
    [
        schema:additionalType apt:numberFirst ;
        schema:value 72
    ] ,
    [
        schema:additionalType apt:road ;
        schema:value <https://linked.data.gov.au/dataset/qldaddr/streetLocality/QLD123756> ; # "Yundah St"
    ] ,
    # ... other properties
.
```

Here the object `QLD12345` is the registered Geographical (Place) name of "Haunted House" which is both assigned to a `Property` (`ParcelAggregate`) or a `Parcel` and referenced in this address as a `propertyName` value.

#### 1.1.3 Address Model & CSDM

For this example Survey Plan:

<img src="module-3/survey-eg-1.png" alt="Survey Example 1" style="width:500px;" />

We have two quoted addresses:

```
212 Victoria Street
```

and

```
2-14 Leicester Street
```

We also have a map coordinate which is "approx. centre of plan", so close to a property centroid.

With this information, we should be able to:

* validate that the addresses exist (if expected to)
* check any known relationship between the addresses
* validate spatial proximity of geocode to plan location
* Auto-generate all subaddresses
    * with a 3D geocode fudging perhaps... +2.5m per level...


A comprehensive CSDM validator (doesn't exist yet) would delegate address validation to the Address Model's validator.


### 1.2 Data Interoperability and Exchange

## 2. Maintenance

### 2.1 Model Extension

#### 2.1.1 Principles

The general principles by which model extensions should occur are:

* **additions are fine**
    * treating this Model as a core, you can add anything else you need to it
    * be sure not to invalidate core patterns
    * reuse background patterns
    * seek to use other common standard patterns
* **change the core only a last resort**
    * to ensure long-term interoperability
    * only all other options have been exhausted
    * "other options" include validator changes and vocabs extension - see below
    * we do expect some, hopefully small, core changes as more jurisdictions come to use the Model
* **publish additions and volicator/vocab changes as a formal _Profile_**
    * this will allow others to potentially reuse your changes, reducing the need for further profiles

#### 2.1.2 Practice

##### 2.1.2.1 Semantic additions

**creating new predicates for new properties/relationships**

e.g. 

```turtle
:address-A ex:mySpecialRelationship :address-B .
```

or

```turtle
:address-A
    ex:mySpecialDataProperty "some value" ;
.
```

Both can be made likely without affecting the core but will require [OWL](https://www.w3.org/TR/owl2-overview/) modelling skills.

If they are only relevant to NSW, these will need their own namespace - see profiles below.

If they are intended for universal use, they should be proposed to the main Model.

**creating specialised classes**

e.g.

```turtle
:RuralAddress rdfs:subclassOf :Address .
```

Specialised classes are exactly equivalent to containers of specialised properties.

All other points as above...

##### 2.1.2.2 Non-semantic additions

If there are properties for objects classes already identified in the model, then key/value pairs for these can be added, bypassing the need for definitional rigour - semantics - however the understanding of these over time will degrade, so this should be seen as a stop-gap measure only.

Qld have done this:

```turtle
PREFIX addr: <https://linked.data.gov.au/dataset/qld-addr/address/>
PREFIX cn: <https://linked.data.gov.au/def/cn/>
PREFIX parcel: <https://linked.data.gov.au/dataset/qld-addr/parcel/>
PREFIX dt: <https://linked.data.gov.au/dataset/qld-addr/datatype/>
PREFIX schema: <https://schema.org/>

addr:4f9a3e3e-7e0c-5176-9f34-958642b1b9ec
    cn:isNameFor parcel:11RP114091 ;
.

parcel:11RP114091
    a <https://linked.data.gov.au/def/addr/AddressableObject> ;
    schema:additionalType <https://linked.data.gov.au/def/go-categories/parcel> ;
    schema:identifier
        "11"^^dt:lot ,
        "11RP114091"^^dt:lotplan ,
        "RP114091"^^dt:plan ;
    schema:name "11RP114091" ;
    schema:additionalProperty
        [
            a schema:PropertyValue ;
            schema:propertyID "parcel_status_code" ;
            schema:value "C" ;
        ] ,
        [
            a schema:PropertyValue ;
            schema:propertyID "parcel_id" ;
            schema:value "917907" ;
        ] ,
        [
            a schema:PropertyValue ;
            schema:propertyID "parcel_create_date" ;
            schema:value "1967-06-20" ;
        ] ;
    cn:hasName addr:4f9a3e3e-7e0c-5176-9f34-958642b1b9ec ;
.
```

Here additional properties for an `AddressableObject`, a `Parcel`, are recorded with just keys, `parcel_status_code`, `parcel_id` etc., to help with transition from current state to future state across the Address & Cadastre systems.

##### 2.1.2.3 Joining to yet other models

All we actually need for this is the "join point" modelled - what Address Model predicate will indicate an identifier in the other model - and that the identifier be an IRI, as per joining to Cadastral data above (Sec 1.1.1).

Best is to position the other model, or at a minumum a shell for it, in relation to the Address Model, within a "Supermodel":

<img src="module-3/qsi-supermodel-models.svg" alt="QSI Supermodel Models" style="width:50%" />

From the [QSI Supermodel](https://linked.data.gov.au/def/qsi-supermodel) specification.

<img src="module-3/other-models.svg" alt="Other Models" style="width:50%" />

### 2.2 Validators

Validators are built seperately from, but related to, the model schema. They allow executable validation.

The Model comes with a base validator - minimal rules that must not be broken for legal model use:

* [Address Model Validator](https://github.com/icsm-au/addr-model/blob/main/validator.ttl)

E.g. validation rule:

```turtle
:addressable-object-present
    a sh:NodeShape ;
    sh:targetClass addr:Address ;
    sh:property [
        sh:path cn:isNameFor ;
        sh:nodeKind sh:IRI ;
        sh:maxCount 1;
        sh:minCount 1;
        sh:message "Each Address must indicate an IRI with the predicate cn:isNameFor" ;
    ] ;
.
```

Queensland's extended validator is listed in the Address Model too. It implements additional, but not conflicting, business rules (incomplete, as of March 2026).

Any other validators may be added - see the profiling below.

### 2.3 Vocabulary Extension

_Adding to the ICSM vocabularies supporting the Address Model is the main way in which the Model is expected to be specialised for different jurisdictions' use._

#### 2.3.1 Principles

##### 2.3.1.1 **Shared Vocabs**: extend/edit ICSM vocabs so other jurisdictions can benefit from your work

We have a series of vocabularies for use with the Model online at <https://icsm.information.qld.gov.au/> (filter by "Address"). Adding to these is recommended to keep the total set of ICSM addressing knowledege in one place.

##### 2.3.1.2 **Extension**: You can add new Concepts vocabs to include representations of new things

e.g. Adding a new Concept for a specialised _Building Name_ part

```
Address Part Types
    |
    + Building Name
```

adding in "Room Name"

```
Address Part Types
    |
    + Building Name
        |
        + Room Name
```

##### 2.3.1.3 **Subsetting**: You can add any number of Collections of terms already existing in vocabs.

e.g. Creating a Collection of Address Part Types allowed to be used in NSW:

```turtle
:nsw-allowed-parts
    a skos:Collection ;
    skos:member
        :addressNumberFirst ,
        :addressNumberSuffix ,
        # ... not all those in the vocab ...
        :waterFeature ;
.
```

##### 2.3.1.4 **Additional Annotations**: You can add alternate labels, notations and scope notes to Concepts

e.g. ACT calling a Subaddress Number a Door Number

```turtle
:subaddressNumber
    skos:prefLabel "Subaddress Number"@en ;
    skos:altLabel "Door Number"^^addr:act ;
    # ... other properties
.
```

Here a new alternative label (`altLabel`) has been applied to the Concept `<https://linked.data.gov.au/def/addr-part-types/subaddressNumber>` and it has been flagged as applying to the ACT (`^^addr:act`) so it can be selected for use when viewed in the ACT.

##### 2.3.1.5 **Recombining Vocabs**: If Extension/Subsetting/Additional Annotations are not enough, whole new vocabs, reusing existing vocab parts and/or adding new ones can be made

Treat this approach with caution! It has a maintenance overhead and can break interoperability.

```
Address Part Types
    |
    + Building Level Type
    |
    ...
```

```
NSW Address Part Types
    |
    + Building Name - Included
    |   |
    |   + Room Name - New
    |
    + Building Level Type - Excluded
    |
    ...

```

#### 2.3.2 Practice

There are many ways to edit elements of standard vocabulary files, such as those relevant to the Address Model. See [KAI's comprehensive vocab documentation](https://docs.kurrawong.ai/concepts/vocabs/introduction/) to learn everything.

In summary:

* [VocExcel](https://docs.kurrawong.ai/products/tools/vocexcel/) for whole new vocab creation
* [VocEdit](https://docs.kurrawong.ai/products/tools/vocedit/)/direct edits for vocab updates

We will do a quick demo of VocExcel and VocEdit using the [Address Part Types](https://linked.data.gov.au/def/addr-part-types) vocab.

The VocExcel demo uses the file `module-3/VocExcel-addr-part-types.xlsx` which was auto-converted to Excel from the point-of-truth file at <https://github.com/icsm-au/icsm-vocabs/tree/main/vocabs/Addresses>.

### 2.4 Profiling

Profiling, for this kind of model, is a formal process of declaring a profile of the specification according to the W3C's [_Profiles Vocabulary_](https://ausbigg.github.io/abis/specification.html#_profiles) (a model):

```turtle
:profile-x
    schema:name "Profile X of the ANZ ICSM Address Model" ;
    prof:isProfileOf <https://linked.data.gov.au/def/addr> ;
.
```

Data that is valid according to Profile X _MUST_ be valida according to the Address Model.

We do not yet have any formalised profiles of the Address Model, but here is an example of a formalised profile of a similarly -constructed model, the [Australian Bidoversity Information Standard (ABIS)](https://linked.data.gov.au/def/abis):

* [BDR profile of ABIS](https://linked.data.gov.au/def/bdr-pr)
    * as it is listed in the ABIS specification: <https://linked.data.gov.au/def/abis#_profiles>.

* Do you REALLY need one?
    * they incur a maintenance overhead
    * try vocab and validator extension first...